# Preparação do corpus de avaliações

## Objetivo

Consolidar as avaliações coletadas, aplicar limpeza, normalização,
tokenização e lematização, e apresentar uma análise descritiva do corpus.

São geradas duas versões: uma com tokens e outra com lemas,
preservando os textos originais em ambas.

Execute as células abaixo na ordem.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import json
from pathlib import Path

import pandas as pd

PASTA_PROJETO = Path("/content/drive/MyDrive/pln_trabalho")
PASTA_RAW = PASTA_PROJETO / "data" / "raw"
PASTA_INTERIM = PASTA_PROJETO / "data" / "interim"

if not PASTA_RAW.is_dir():
    raise FileNotFoundError(f"Pasta não encontrada: {PASTA_RAW}")

ARQUIVO_SAIDA = PASTA_INTERIM / "avaliacoes_consolidadas.csv"

print(f"Origem: {PASTA_RAW}")
print(f"Destino: {ARQUIVO_SAIDA}")

Origem: /content/drive/MyDrive/pln_trabalho/data/raw
Destino: /content/drive/MyDrive/pln_trabalho/data/interim/avaliacoes_consolidadas.csv


In [5]:
arquivos = sorted(PASTA_RAW.glob("*_nota_[1-5].json"))

if len(arquivos) != 50:
    raise ValueError(f"Esperados 50 arquivos; encontrados {len(arquivos)}.")

registros = []

for arquivo in arquivos:
    with arquivo.open("r", encoding="utf-8") as entrada:
        pacote = json.load(entrada)

    registros.extend(pacote["avaliacoes"])

df = pd.DataFrame(registros)

campos_obrigatorios = {"reviewId", "content", "score", "app_id", "app_name"}

if not campos_obrigatorios.issubset(df.columns):
    raise ValueError("Há campos obrigatórios ausentes na tabela.")

if len(df) != 5000:
    raise ValueError(f"Esperadas 5.000 avaliações; encontradas {len(df)}.")

if df["app_id"].nunique() != 10:
    raise ValueError("A tabela não contém exatamente dez aplicativos.")

for campo in ["reviewId", "content"]:
    validos = df[campo].map(
        lambda valor: isinstance(valor, str) and bool(valor.strip())
    )
    if not validos.all():
        raise ValueError(f"Há valores vazios ou inválidos em '{campo}'.")

if df.duplicated(subset=["app_id", "reviewId"]).any():
    raise ValueError("Há IDs repetidos no mesmo aplicativo.")

notas_validas = df["score"].map(
    lambda valor: type(valor) is int and valor in range(1, 6)
)
if not notas_validas.all():
    raise ValueError("Há notas inválidas.")

distribuicao = pd.crosstab(df["app_id"], df["score"]).reindex(
    columns=[1, 2, 3, 4, 5],
    fill_value=0,
)

if not distribuicao.eq(100).all().all():
    raise ValueError("Há quantidades diferentes de 100 por aplicativo e nota.")

print(f"Arquivos lidos: {len(arquivos)}")
print(f"Avaliações: {len(df)}")
print(f"Aplicativos: {df['app_id'].nunique()}")

Arquivos lidos: 50
Avaliações: 5000
Aplicativos: 10


In [7]:
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)

df.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8",
)

print(f"CSV consolidado salvo: {ARQUIVO_SAIDA}")
print(f"Total de avaliações: {len(df)}")

CSV consolidado salvo: /content/drive/MyDrive/pln_trabalho/data/interim/avaliacoes_consolidadas.csv
Total de avaliações: 5000


## Limpeza dos textos

Padronização de espaços em branco e remoção
de espaços no início e no fim dos comentários.


In [8]:
import re


def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


df["texto_limpo"] = df["content"].map(limpar_texto)

alterados = df["content"].ne(df["texto_limpo"])

print(f"Textos conferidos: {len(df)}")
print(f"Textos alterados pela limpeza: {alterados.sum()}")



Textos conferidos: 5000
Textos alterados pela limpeza: 1


## Normalização dos textos

Conversão dos textos limpos para minúsculas, armazenando o resultado
na coluna `texto_normalizado`.

In [9]:
def normalizar_texto(texto):
    return texto.lower()


df["texto_normalizado"] = df["texto_limpo"].map(normalizar_texto)

alterados = df["texto_limpo"].ne(df["texto_normalizado"])

print(f"Textos conferidos: {len(df)}")
print(f"Textos alterados pela normalização: {alterados.sum()}")



Textos conferidos: 5000
Textos alterados pela normalização: 3962


## Tokenização

Os textos são tokenizados com o spaCy. Os documentos resultantes são
preservados em memória para que a lematização utilize os mesmos tokens.


###Instalação do modelo

In [10]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "spacy"],
    check=True
)

subprocess.run(
    [sys.executable, "-m", "spacy", "download", "pt_core_news_sm"],
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'spacy', 'download', 'pt_core_news_sm'], returncode=0)

###Carregamento

In [11]:
import spacy

nlp = spacy.load(
    "pt_core_news_sm",
    disable=["ner", "parser"]
)

print(f"spaCy: {spacy.__version__}")
print(f"Modelo: {nlp.meta['name']}")
print(f"Versão do modelo: {nlp.meta['version']}")
print(f"pandas: {pd.__version__}")

spaCy: 3.8.16
Modelo: core_news_sm
Versão do modelo: 3.8.0
pandas: 2.2.3


###Aplicação

In [12]:
df = df.drop(columns=["lemas"], errors="ignore")

documentos = [
    nlp.make_doc(texto)
    for texto in df["texto_limpo"]
]

df["tokens"] = [
    [
        token.text.lower()
        for token in documento
        if not token.is_space
    ]
    for documento in documentos
]

print(f"Avaliações tokenizadas: {len(df)}")
print(f"Total de tokens: {df['tokens'].map(len).sum()}")

Avaliações tokenizadas: 5000
Total de tokens: 288681


## Lematização

O modelo de português `pt_core_news_sm` processa os documentos já
tokenizados, considerando o contexto de cada avaliação.

Como o modelo pode produzir erros, são mantidas duas versões do corpus:
uma com os tokens e outra com os lemas. O texto original é preservado
em ambas.

### Aplicação no corpus

In [13]:
listas_lemas = []

# Recebe os documentos já tokenizados, sem recriar a tokenização.
for documento in nlp.pipe(documentos, batch_size=64):
    listas_lemas.append([
        token.lemma_.lower()
        for token in documento
        if not token.is_space
    ])

# A versão lematizada contém lemas em vez da coluna tokens.
df_lematizado = df.drop(columns=["tokens"]).copy()
df_lematizado["lemas"] = listas_lemas

print(f"Avaliações lematizadas: {len(df_lematizado)}")

Avaliações lematizadas: 5000


###Exemplos

In [14]:
for tokens, lemas in zip(df["tokens"], df_lematizado["lemas"]):
    if len(tokens) != len(lemas):
        raise ValueError("Quantidade de tokens e lemas divergente.")

print("Conferência concluída: um lema para cada token.")

posicoes = (
    df.reset_index(drop=True)
    .sample(n=min(3, len(df)), random_state=42)
    .index
)

for posicao in posicoes:
    print("\nTEXTO:", df.iloc[posicao]["texto_limpo"])
    print("TOKENS:", df.iloc[posicao]["tokens"])
    print("LEMAS:", df_lematizado.iloc[posicao]["lemas"])

Conferência concluída: um lema para cada token.

TEXTO: Achava a melhor IA que existia, assinei a versão PRO e ficou simplesmente impossível de usar. Mando uma mensagem e ele praticamente já consome todo o limite de uso. IMPOSSÍVEL usar assim! A sensação que tenho é que na versão free eu fazia muito mais coisas. É extremamente irritante e difícil de entender esse consumo rápido de tokens, mesmo você tentando usar da maneira mais econômica possível.
TOKENS: ['achava', 'a', 'melhor', 'ia', 'que', 'existia', ',', 'assinei', 'a', 'versão', 'pro', 'e', 'ficou', 'simplesmente', 'impossível', 'de', 'usar', '.', 'mando', 'uma', 'mensagem', 'e', 'ele', 'praticamente', 'já', 'consome', 'todo', 'o', 'limite', 'de', 'uso', '.', 'impossível', 'usar', 'assim', '!', 'a', 'sensação', 'que', 'tenho', 'é', 'que', 'na', 'versão', 'free', 'eu', 'fazia', 'muito', 'mais', 'coisas', '.', 'é', 'extremamente', 'irritante', 'e', 'difícil', 'de', 'entender', 'esse', 'consumo', 'rápido', 'de', 'tokens', ',', 'mes

### Salvamento


In [15]:
PASTA_PROCESSED = PASTA_INTERIM.parent / "processed"
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

ARQUIVO_SEM_LEMAS = PASTA_PROCESSED / "avaliacoes_preparadas.json"
ARQUIVO_COM_LEMAS = PASTA_PROCESSED / "avaliacoes_lematizadas.json"

df.to_json(
    ARQUIVO_SEM_LEMAS,
    orient="records",
    force_ascii=False,
    indent=2
)

df_lematizado.to_json(
    ARQUIVO_COM_LEMAS,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"Arquivo com tokens: {ARQUIVO_SEM_LEMAS}")
print(f"Arquivo com lemas: {ARQUIVO_COM_LEMAS}")

Arquivo com tokens: /content/drive/MyDrive/pln_trabalho/data/processed/avaliacoes_preparadas.json
Arquivo com lemas: /content/drive/MyDrive/pln_trabalho/data/processed/avaliacoes_lematizadas.json


## Análise descritiva do corpus

As tabelas a seguir apresentam a distribuição das avaliações, o comprimento
dos textos e exemplos das transformações realizadas na preparação.

### 1. Distribuição por aplicativo e nota

Quantidade de avaliações por aplicativo e nota, para conferir as cotas
definidas na coleta: 100 avaliações por nota em cada aplicativo.

Essa distribuição foi definida previamente e não representa a distribuição
natural das notas na Google Play nem o equilíbrio entre emoções.

In [42]:
from IPython.display import display

distribuicao = pd.crosstab(
    df["app_name"],
    df["score"]
).reindex(columns=[1, 2, 3, 4, 5], fill_value=0)

distribuicao.index.name = "Aplicativo"
distribuicao.columns.name = "Nota"
distribuicao["Total"] = distribuicao.sum(axis=1)

display(distribuicao)

Nota,1,2,3,4,5,Total
Aplicativo,,,,,,
ChatGPT,100,100,100,100,100,500
Claude,100,100,100,100,100,500
DeepSeek,100,100,100,100,100,500
Dola,100,100,100,100,100,500
Google Gemini,100,100,100,100,100,500
Grok,100,100,100,100,100,500
Luzia,100,100,100,100,100,500
Meta AI,100,100,100,100,100,500
Nova,100,100,100,100,100,500


### 2. Comprimento das avaliações

Comprimento mínimo, mediano e máximo dos textos originais por aplicativo,
medido em caracteres.

A mediana indica o comprimento central da distribuição. Textos mais longos
podem fornecer mais contexto, mas não são necessariamente mais informativos
ou emocionalmente mais expressivos.

In [43]:
comprimentos = (
    df.assign(caracteres=df["content"].str.len())
    .groupby("app_name")["caracteres"]
    .agg(
        Minimo="min",
        Mediana="median",
        Maximo="max"
    )
)

comprimentos.index.name = "Aplicativo"

display(comprimentos)

,Minimo,Mediana,Maximo
Aplicativo,,,
ChatGPT,122,389.5,500
Claude,36,265.0,500
DeepSeek,48,313.0,500
Dola,70,302.0,500
Google Gemini,89,334.0,500
Grok,83,327.5,500
Luzia,38,252.0,500
Meta AI,20,130.0,500
Nova,15,206.0,500


### 3. Exemplos de limpeza e normalização

Comparação entre o texto original, o texto após o tratamento dos espaços
e o texto convertido para minúsculas.

São priorizados exemplos que sofreram alterações. A limpeza pode manter
o texto idêntico ao original quando não existem espaços a corrigir.
Acentos, pontuação e emojis são preservados.

In [44]:
mudou_limpeza = df["content"].ne(df["texto_limpo"])
mudou_normalizacao = df["texto_limpo"].ne(df["texto_normalizado"])

exemplos = pd.concat([
    df.loc[mudou_limpeza],
    df.loc[~mudou_limpeza & mudou_normalizacao],
    df.loc[~mudou_limpeza & ~mudou_normalizacao]
]).head(3)

comparacao = exemplos[
    ["content", "texto_limpo", "texto_normalizado"]
].rename(columns={
    "content": "Original",
    "texto_limpo": "Após limpeza",
    "texto_normalizado": "Após normalização"
})

with pd.option_context("display.max_colwidth", None):
    display(comparacao.reset_index(drop=True))

,Original,Após limpeza,Após normalização
0,"Prezada equipe de desenvolvimento do DeepSeek, Gostaria de parabenizá-los primeiramente pelo excelente trabalho. Utilizo o DeepSeek com frequência e Admiro a inteligência e a Qualidade gratuita do assistente. No entanto, gostaria de Sugerir, da forma mais cordial, a implementação de um Recurso que considero essencial para a Acessibilidade e para uma experiência de usuário de ponta: o Modo de conversa fluida e ""Mãos Livres' pois, é assim como eu interajo com a IA Gemini da Google...","Prezada equipe de desenvolvimento do DeepSeek, Gostaria de parabenizá-los primeiramente pelo excelente trabalho. Utilizo o DeepSeek com frequência e Admiro a inteligência e a Qualidade gratuita do assistente. No entanto, gostaria de Sugerir, da forma mais cordial, a implementação de um Recurso que considero essencial para a Acessibilidade e para uma experiência de usuário de ponta: o Modo de conversa fluida e ""Mãos Livres' pois, é assim como eu interajo com a IA Gemini da Google...","prezada equipe de desenvolvimento do deepseek, gostaria de parabenizá-los primeiramente pelo excelente trabalho. utilizo o deepseek com frequência e admiro a inteligência e a qualidade gratuita do assistente. no entanto, gostaria de sugerir, da forma mais cordial, a implementação de um recurso que considero essencial para a acessibilidade e para uma experiência de usuário de ponta: o modo de conversa fluida e ""mãos livres' pois, é assim como eu interajo com a ia gemini da google..."
1,"Aff, sério, Perplexity? 😑 Todo mundo fala que é “top”, mas juro que só dá dor de cabeça. O app trava do nada, as respostas às vezes parecem que foram geradas por uma máquina com sono profundo, e não dá nem pra confiar que as informações estejam certas. 😒 Pior é quando você quer algo rápido e ele insiste em enrolar ou não entende nada direito. Fora que a interface podia ser bem mais intuitiva, porque se não você fica perdido tentando achar uma função simples. Aff, podia ser tão bom, mas parece","Aff, sério, Perplexity? 😑 Todo mundo fala que é “top”, mas juro que só dá dor de cabeça. O app trava do nada, as respostas às vezes parecem que foram geradas por uma máquina com sono profundo, e não dá nem pra confiar que as informações estejam certas. 😒 Pior é quando você quer algo rápido e ele insiste em enrolar ou não entende nada direito. Fora que a interface podia ser bem mais intuitiva, porque se não você fica perdido tentando achar uma função simples. Aff, podia ser tão bom, mas parece","aff, sério, perplexity? 😑 todo mundo fala que é “top”, mas juro que só dá dor de cabeça. o app trava do nada, as respostas às vezes parecem que foram geradas por uma máquina com sono profundo, e não dá nem pra confiar que as informações estejam certas. 😒 pior é quando você quer algo rápido e ele insiste em enrolar ou não entende nada direito. fora que a interface podia ser bem mais intuitiva, porque se não você fica perdido tentando achar uma função simples. aff, podia ser tão bom, mas parece"
2,"Uma estrela e olhe lá. Pelas propagandas parecia a oitava maravilha do mundo, capaz de superar o ChatGPT e o Google. Na prática, não entende nem um simples “oi” e responde tudo errado. Se vende como a evolução da inteligência artificial, mas entrega confusão, respostas absurdas e sem contexto. Baixei esperando um gênio digital e recebi um erro ambulante. Péssimo é pouco.","Uma estrela e olhe lá. Pelas propagandas parecia a oitava maravilha do mundo, capaz de superar o ChatGPT e o Google. Na prática, não entende nem um simples “oi” e responde tudo errado. Se vende como a evolução da inteligência artificial, mas entrega confusão, respostas absurdas e sem contexto. Baixei esperando um gênio digital e recebi um erro ambulante. Péssimo é pouco.","uma estrela e olhe lá. pelas propagandas parecia a oitava maravilha do mundo, capaz de superar o chatgpt e o google. na prática, não entende nem um simples “oi” e responde tudo errado. se vende como a evolução da inteligência arti

 4. Exemplos de lematização

Comparação entre tokens e lemas das mesmas avaliações. São exibidos
até 20 pares distintos que sofreram alterações.

In [45]:
pares = df[
    ["app_id", "reviewId", "tokens"]
].merge(
    df_lematizado[["app_id", "reviewId", "lemas"]],
    on=["app_id", "reviewId"],
    validate="one_to_one"
)

alteracoes = []
pares_vistos = set()

for tokens, lemas in zip(pares["tokens"], pares["lemas"]):
    if len(tokens) != len(lemas):
        raise ValueError("Quantidade de tokens e lemas divergente.")

    for token, lema in zip(tokens, lemas):
        par = (token, lema)

        if token != lema and par not in pares_vistos:
            alteracoes.append({"Token": token, "Lema": lema})
            pares_vistos.add(par)

        if len(alteracoes) == 20:
            break

    if len(alteracoes) == 20:
        break

display(pd.DataFrame(alteracoes, columns=["Token", "Lema"]))

,Token,Lema
0,fala,falar
1,é,ser
2,dá,dar
3,trava,trar
4,do,de o
5,as,o
6,respostas,resposta
7,às,a o
8,vezes,vez
9,parecem,parecer


In [16]:
import sys
from importlib.metadata import version

print(f"Python: {sys.version.split()[0]}")

for pacote in ["pandas", "spacy", "pt-core-news-sm"]:
    print(f"{pacote}=={version(pacote)}")

Python: 3.13.15
pandas==2.2.3
spacy==3.8.16
pt-core-news-sm==3.8.0
